In [ ]:
# Imagem embutida no código (foto original em base64) - não precisa fazer upload
import base64
IMG_B64 = (
    "..."
)
open('imagem.jpg', 'wb').write(base64.b64decode(IMG_B64))
print('imagem.jpg criada')


In [ ]:
import math, cv2
import numpy as np
import matplotlib.pyplot as plt

I = cv2.cvtColor(cv2.imread("imagem.jpg"), cv2.COLOR_BGR2GRAY)

print("Largura:", I.shape[1], "| Altura:", I.shape[0], "| Total de pixels:", I.size)
print("Menor:", I.min(), "| Maior:", I.max(), "| Média:", round(I.mean(), 2))

def gaussiano(su, sv):
    """Kernel gaussiano normalizado (u = horizontal, v = vertical)."""
    a, b = math.ceil(3*su), math.ceil(3*sv)          # n = 2*ceil(3σ)+1
    U, V = np.meshgrid(np.arange(-a, a+1), np.arange(-b, b+1))
    G = np.exp(-0.5 * (U**2/su**2 + V**2/sv**2)) / (2*np.pi*su*sv)
    return G / G.sum(), U, V

def filtrar(img, K):
    return cv2.filter2D(img, -1, K)

plt.imshow(I, cmap="gray"); plt.title("Original (cinza)"); plt.axis("off"); plt.show()

In [ ]:
# Kernel de média 3x3
KM = np.ones((3, 3)) / 9
print(KM)
print("Soma:", KM.sum())

fig, ax = plt.subplots(1, 2, figsize=(10, 7))
ax[0].imshow(I, cmap="gray"); ax[0].set_title("Original")
ax[1].imshow(filtrar(I, KM), cmap="gray"); ax[1].set_title("Média 3x3")
plt.show()

In [ ]:
# Gaussiano isotrópico σ = 1
K1, U, V = gaussiano(1, 1)
print(np.round(K1, 4))
print("Soma:", K1.sum(), "| Tamanho:", K1.shape)

fig = plt.figure(figsize=(14, 5))
ax = fig.add_subplot(1, 3, 1); ax.imshow(K1); ax.set_title("Pesos 2D")
ax = fig.add_subplot(1, 3, 2, projection="3d"); ax.plot_surface(U, V, K1, cmap="viridis"); ax.set_title("Superfície 3D")
ax = fig.add_subplot(1, 3, 3); ax.imshow(filtrar(I, K1), cmap="gray"); ax.set_title("Filtrada σ=1"); ax.axis("off")
plt.show()

In [ ]:
# Influência de σ
sigmas = [0.8, 1.5, 3.0]
fig, ax = plt.subplots(1, 4, figsize=(20, 7))
ax[0].imshow(I, cmap="gray"); ax[0].set_title("Original")
for a, s in zip(ax[1:], sigmas):
    K, _, _ = gaussiano(s, s)
    print(f"σ={s}: kernel {K.shape}, soma={K.sum():.4f}")
    a.imshow(filtrar(I, K), cmap="gray"); a.set_title(f"σ={s}")
plt.show()

# Média x Gaussiano com a mesma dimensão (11x11, σ=1.5)
Kg, _, _ = gaussiano(1.5, 1.5)
n = Kg.shape[0]
Km = np.ones((n, n)) / n**2
fig, ax = plt.subplots(1, 3, figsize=(16, 7))
ax[0].imshow(I, cmap="gray"); ax[0].set_title("Original")
ax[1].imshow(filtrar(I, Km), cmap="gray"); ax[1].set_title(f"Média {n}x{n}")
ax[2].imshow(filtrar(I, Kg), cmap="gray"); ax[2].set_title(f"Gaussiano {n}x{n}")
plt.show()

In [ ]:
# Kernels não isotrópicos
K_a, Ua, Va = gaussiano(3, 0.8)    # σu > σv
K_b, Ub, Vb = gaussiano(0.8, 3)    # σv > σu
print("Somas:", K_a.sum(), K_b.sum())

fig = plt.figure(figsize=(12, 8))
for i, (K, U_, V_, t) in enumerate([(K_a, Ua, Va, "σu=3, σv=0.8"), (K_b, Ub, Vb, "σu=0.8, σv=3")]):
    ax = fig.add_subplot(2, 2, i+1); ax.imshow(K); ax.set_title(t)
    ax = fig.add_subplot(2, 2, i+3, projection="3d"); ax.plot_surface(U_, V_, K, cmap="viridis")
plt.show()

# Aplicação
Ki, _, _ = gaussiano(1.5, 1.5)
fig, ax = plt.subplots(1, 4, figsize=(20, 7))
for a, (img, t) in zip(ax, [(I, "Original"), (filtrar(I, Ki), "Isotrópico σ=1.5"),
                            (filtrar(I, K_a), "Iuv1: σu=3, σv=0.8"),
                            (filtrar(I, K_b), "Iuv2: σu=0.8, σv=3")]):
    a.imshow(img, cmap="gray"); a.set_title(t)
plt.show()

In [ ]:
# Perfis dos kernels
casos = {"Isotrópico σ=1.5": Ki, "σu=3, σv=0.8": K_a, "σu=0.8, σv=3": K_b}

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for nome, K in casos.items():
    ax[0].plot(np.arange(K.shape[1]) - K.shape[1]//2, K[K.shape[0]//2, :], "o-", label=nome)
    ax[1].plot(np.arange(K.shape[0]) - K.shape[0]//2, K[:, K.shape[1]//2], "o-", label=nome)
ax[0].set_title("Linha central (direção u)")
ax[1].set_title("Coluna central (direção v)")
for a in ax: a.legend(); a.grid(alpha=.3)
plt.show()